### Imports
Please follow the README's instructions under 'Installations' to ensure you have the appropriate dependencies to run this notebook.

In [ ]:
import os
import numpy as np
import skimage.io as io
import cv2 as cv
import random
import shutil

cwd = os.getcwd()
parent_dir = os.path.abspath(os.path.join(cwd, os.pardir))

# Import OpenSlide
OPENSLIDE_PATH = os.path.join(parent_dir, 'openslide-bin-4.0.0.10-windows-x64\\bin')

if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide
import xml.etree.cElementTree as ET

### Utils

In [ ]:

def parse_xml(anno_path, id_num, factor):
    """Parse XML annotation file and extract coordinates for a specific ID."""
    tree = ET.ElementTree(file=anno_path)
    annolist = {}
    root = tree.getroot()
    
    for annotation in root.iter('Annotation'):
        if annotation.attrib.get('Id') == str(id_num):
            i = 0
            for region in annotation.findall(".//Region"):
                vasc = []
                for vertex in region.findall(".//Vertex"):
                    try:
                        x = float(vertex.attrib.get("X"))
                        y = float(vertex.attrib.get("Y"))
                        vasc.append((int(x / factor), int(y / factor)))
                    except Exception as e:
                        print(f"Error parsing coordinates: {e}")
                        continue
                annolist[i] = vasc
                i += 1

    if len(annolist) == 0:
        anno_filename = os.path.basename(anno_path)
        print(f'There are no annotations for ID {id_num} in {anno_filename}')

    return annolist


def is_mostly_white(image, threshold=0.7):
    """Check if an image is mostly white based on threshold."""
    # Convert the image to grayscale
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    # Apply threshold to identify white pixels
    _, binary = cv.threshold(gray, 220, 255, cv.THRESH_BINARY)
    # Calculate the percentage of white pixels
    white_percentage = np.sum(binary == 255) / binary.size
    return white_percentage >= threshold


def extract_patches_with_padding(wsi, annolist, patch_size, factor, mlevel,
                                 padding_percent=0, overlap_percent=0,
                                 min_anno_overlap=0.7):
    """Extract patches from WSI based on annotations."""
    patches = []
    patch_width, patch_height = patch_size
    padding = int(min(patch_width, patch_height) * (padding_percent / 100))
    overlap = int(min(patch_width, patch_height) * (overlap_percent / 100))
    
    for i, coords in annolist.items():

        # convert annotation coordinates to numpy array
        annotation = np.array(coords, dtype=np.int32)

        # find bounding rectangle for the annotation
        x, y, w, h = cv.boundingRect(annotation)
        
        # calculate patch grid within bounding rectangle
        num_patches_x = max(1, (w - overlap) // (patch_width - overlap))
        num_patches_y = max(1, (h - overlap) // (patch_height - overlap))

        # create a dummy mask of the anotation area to check patch overlap
        anno_mask = np.zeros((h, w), dtype=np.uint8)

        # shift annotation coords relative to bounding box
        shifted_anno = annotation.copy()
        shifted_anno[:, 0] -= x # subtract from all x values
        shifted_anno[:, 1] -= y # subtract from all y values

        # fill annotation mask
        cv.fillPoly(anno_mask, [shifted_anno], 1)

        for y_idx in range(num_patches_y):
            for x_idx in range(num_patches_x):
                # Calculate patch position within bounding box
                patch_x = x_idx * (patch_width - overlap)
                patch_y = y_idx * (patch_height - overlap)

                # Ensure we don't go outside bounding box
                patch_end_x = min(patch_x + patch_width, w)
                patch_end_y = min(patch_y + patch_height, h)

                # extract patch region from annotation mask
                patch_anno_region = anno_mask[patch_y:patch_end_y, patch_x:patch_end_x]

                # calculate overlap percentage
                anno_area = patch_width * patch_height
                overlap_area = np.sum(patch_anno_region)
                overlap_percentage = overlap_area / anno_area if anno_area > 0 else 0

                # skip patch extraction if overlap is insufficient
                if overlap_percentage < min_anno_overlap:
                    continue

                # calculate actual WSI coordinates (with padding)
                actual_x = max(x + patch_x - padding, 0)
                actual_y = max(y + patch_y - padding, 0)
                actual_end_x = min(actual_x + patch_width, wsi.level_dimensions[0][0])
                actual_end_y = min(actual_y + patch_height, wsi.level_dimensions[0][1])
                
                # adjust for padding at edges
                actual_patch_width = actual_end_x - actual_x
                actual_patch_height = actual_end_y - actual_y
                
                # read patch from WSI
                patch_img_rgba = np.asarray(
                    wsi.read_region(
                        (actual_x * factor, actual_y * factor),
                        mlevel,
                        (actual_patch_width, actual_patch_height)
                    )
                )
                patch_img = patch_img_rgba[:, :, :3]
                
                # check if patch is mostly background
                if not is_mostly_white(patch_img):
                    patches.append(patch_img)
    
    return patches


def save_patches(patches, output_dir, slide_name, folder_name, patch_size):
    """Save extracted patches to disk."""
    os.makedirs(output_dir, exist_ok=True)
    patch_w, _ = patch_size
    
    for i, patch in enumerate(patches):
        save_as = os.path.join(
            output_dir, 
            f'{slide_name}_{folder_name}_{patch_w}_patch_position_{i}.tif'
        )
        io.imsave(save_as, patch, check_contrast=False)


def generate_patches_for_slide(wsi_path, xml_path, output_base_dir, 
                               patch_sizes, id_folders, factor, mlevel):
    """Main function to generate patches for a single slide."""
    # Load WSI
    wsi = openslide.OpenSlide(wsi_path)
    slidename = os.path.splitext(os.path.basename(wsi_path))[0]
    
    # Loop through each annotation category
    for i, folder in enumerate(id_folders):
        if folder == 'skip':
            continue
            
        anno_id = i + 1
        slide_dir = os.path.join(output_base_dir, folder, slidename)
        
        # Parse annotations
        annolist = parse_xml(xml_path, anno_id, factor)
        
        # Skip if no annotations found
        if not annolist:
            continue
            
        # Generate patches for each patch size
        for patch_size in patch_sizes:
            patches = extract_patches_with_padding(
                wsi, annolist, patch_size, factor, mlevel,
                padding_percent=0, overlap_percent=0
            )
            
            # Save patches
            patch_size_dir = os.path.join(slide_dir, str(patch_size[0]))
            save_patches(
                patches, patch_size_dir, slidename, folder, patch_size
            )
    
    return slidename

def run_patch_extraction(
    folder_path,
    id_folders,
    patch_sizes=[(48, 48), (256, 256)],
    mlevel=0,
):
    """
    Run patch extraction for all .svs slides in a folder.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing .svs WSI files and matching .xml annotations.
    id_folders : list of str
        Annotation category folder names, ordered by annotation ID.
        Use 'skip' to ignore an annotation ID (e.g. ['id1_S', 'skip', 'id3_O']).
    patch_sizes : list of tuple, optional
        List of (width, height) patch sizes to extract. Default: [(48, 48), (256, 256)].
    mlevel : int, optional
        OpenSlide pyramid level to read from. Default: 0 (highest resolution).

    Returns
    -------
    list of str
        Slide names that were successfully processed.
    """
    factor = 2 ** mlevel

    # Create base output directories for each annotation category
    for folder in id_folders:
        if folder != 'skip':
            os.makedirs(os.path.join(folder_path, folder), exist_ok=True)

    processed_slides = []

    # Loop through each .svs file in the folder
    for file_name in os.listdir(folder_path):
        if not file_name.endswith(".svs"):
            continue

        wsi_path = os.path.join(folder_path, file_name)
        xml_path = os.path.join(folder_path, f"{os.path.splitext(file_name)[0]}.xml")

        if not os.path.exists(xml_path):
            print(f"Warning: No XML file found for {file_name}, skipping.")
            continue

        print(f"Processing {file_name}...")
        try:
            slidename = generate_patches_for_slide(
                wsi_path, xml_path, folder_path,
                patch_sizes, id_folders, factor, mlevel
            )
            print(f"  Completed: {slidename}")
            processed_slides.append(slidename)
        except Exception as e:
            print(f"  Error processing {file_name}: {e}")

    print(f"\nDone. Successfully processed {len(processed_slides)} slide(s).")
    return processed_slides


# --- CONFIGURATION & EXECUTION ----------------------------------------
if __name__ == "__main__":
    run_patch_extraction(
        folder_path=r"E:\[PROJ]_TILseg_backups\TILseg_TCGA\annotations",  # TODO
        id_folders=['id1_S', 'id2_E', 'id3_O'],                           # TODO
        patch_sizes=[(48, 48), (256, 256)],
        mlevel=0,
    )


### Execution: Extract Training Patches
Annotations must be drawn and exported from Aperio ImageScope as .xml files, with each .xml file sharing the same base name as its corresponding .svs slide (e.g., slide_001.svs → slide_001.xml). Both files must reside in the same directory. 

Within ImageScope, each annotation category (e.g., stroma, epithelium, other) must be assigned to a separate, uniquely numbered Annotation ID (i.e. annotation Layer), accessible via the Annotations panel. The id_folders list defined in the configuration below must be ordered to match these IDs sequentially — for example, if stroma annotations are drawn in Layer 1 and epithelium is in Layer 2, then id_folders = ['id1_S', 'id2_E', ...]. Use 'skip' at any index position to ignore that annotation ID during extraction. Annotation coordinates are stored by ImageScope in full-resolution pixel space (level 0), which this pipeline handles automatically based on the mlevel parameter.

In [ ]:
# CONFIGURATION --------------------------------------------------------
mlevel = 0  # reading the WSI at level 1 (40x)
factor = 2 ** mlevel
patch_sizes = [(48, 48), (256, 256)]

# Set the path to the folder containing WSIs and XML annotations
folder_path = r"path/to/folder" ## TODO

# Define annotation categories
id_folders = ['id1_S', 'id2_E', 'id3_O'] ## TODO
# ----------------------------------------------------------------------

#### Here is an example of how the directory structure created looks like after extracting patches:

In [ ]:
# path/to/input/folder
# ├── id1_stroma\
# │   └── slide_name\
# │       ├── 48\      ← 48x48 patches
# │       └── 256\     ← 256x256 patches
# ├── id2_epithelium\
# │   └── slide_name\
# │       ├── 48\
# │       └── 256\
# └── id3_other\
#     └── slide_name\
#         ├── 48\
#         └── 256\